### Installation

In [23]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [24]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/llama-2-13b-bnb-4bit",
    "unsloth/codellama-34b-bnb-4bit",
    "unsloth/tinyllama-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit", # New Google 6 trillion tokens model 2.5x faster!
    "unsloth/gemma-2b-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/tinyllama-bnb-4bit", # "unsloth/tinyllama" for 16bit loading
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [25]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Currently only supports dropout = 0
    bias = "none",    # Currently only supports bias = "none"
    use_gradient_checkpointing = False, # @@@ IF YOU GET OUT OF MEMORY - set to True @@@
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

### Data Prep

In [26]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset

# Load the local Alpaca-formatted JSON file
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("newadays/alpaca_ig_post", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

createkap_training_v2_alpaca.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1138 [00:00<?, ? examples/s]

Map:   0%|          | 0/1138 [00:00<?, ? examples/s]

### Training

In [27]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = True, # Packs short sequences together to save time!
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1138 [00:00<?, ? examples/s]

In [28]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
2.85 GB of memory reserved.


In [29]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,138 | Num Epochs = 3 | Total steps = 429
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 25,231,360 of 1,125,279,744 (2.24% trained)


Step,Training Loss
1,2.460600
2,2.498400
3,2.580800
4,2.609000
5,2.602400
6,2.697200
7,2.650900
8,2.659900
9,2.676800
10,2.611400


In [30]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

905.8367 seconds used for training.
15.1 minutes used for training.
Peak reserved memory = 2.85 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 19.57 %.
Peak reserved memory for training % of max memory = 0.0 %.


### Inference

In [31]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Generate an creative Instagram caption described in the context provided in triple backticks for a brand called Chernov Team Realtor located in Los Angeles California with emojis.\n\nPlease follow the steps below to perform the task:\n1 - Add important information about the real estate that will captivate readers. Do not add a description about the image.\n2 - Itemize the good features of a good property in a List\n3 - Include the location of the listing\n4 - Remember to add hashtags related to the product at the end of the caption in a separate line.\n5 - Not more than 100 words\n\nPlease remember to follow these important guidelines:\n- Remember to use proper grammar throughout the caption\n- Keep the caption short and to the point.\n- Use an enthusiastic tone throughout the caption.\n- Create a Sense of Urgency throughout the caption.\n- Remember to itemize the features in a List\n\n```\na white house with the words just listed above it\n```",
        "", # input
        "", # output - leave blank for generation
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
tokenizer.batch_decode(outputs)

['<s> Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nGenerate an creative Instagram caption described in the context provided in triple backticks for a brand called Chernov Team Realtor located in Los Angeles California with emojis.\n\nPlease follow the steps below to perform the task:\n1 - Add important information about the real estate that will captivate readers. Do not add a description about the image.\n2 - Itemize the good features of a good property in a List\n3 - Include the location of the listing\n4 - Remember to add hashtags related to the product at the end of the caption in a separate line.\n5 - Not more than 100 words\n\nPlease remember to follow these important guidelines:\n- Remember to use proper grammar throughout the caption\n- Keep the caption short and to the point.\n- Use an enthusiastic tone throughout the caption.\n- Create a Sense 

In [32]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Generate an creative Instagram caption described in the context provided in triple backticks for a brand called Chernov Team Realtor located in Los Angeles California with emojis.\n\nPlease follow the steps below to perform the task:\n1 - Add important information about the real estate that will captivate readers. Do not add a description about the image.\n2 - Itemize the good features of a good property in a List\n3 - Include the location of the listing\n4 - Remember to add hashtags related to the product at the end of the caption in a separate line.\n5 - Not more than 100 words\n\nPlease remember to follow these important guidelines:\n- Remember to use proper grammar throughout the caption\n- Keep the caption short and to the point.\n- Use an enthusiastic tone throughout the caption.\n- Create a Sense of Urgency throughout the caption.\n- Remember to itemize the features in a List\n\n```\na white house with the words just listed above it\n```",
        "", # input
        "", # output - leave blank for generation
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 512)

<s> Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Generate an creative Instagram caption described in the context provided in triple backticks for a brand called Chernov Team Realtor located in Los Angeles California with emojis.

Please follow the steps below to perform the task:
1 - Add important information about the real estate that will captivate readers. Do not add a description about the image.
2 - Itemize the good features of a good property in a List
3 - Include the location of the listing
4 - Remember to add hashtags related to the product at the end of the caption in a separate line.
5 - Not more than 100 words

Please remember to follow these important guidelines:
- Remember to use proper grammar throughout the caption
- Keep the caption short and to the point.
- Use an enthusiastic tone throughout the caption.
- Create a Sense of Urgency through

### Save Model

In [ ]:
model.save_pretrained("tinyllama_lora_ig_post")  # Local saving
tokenizer.save_pretrained("tinyllama_lora_ig_post")
model.push_to_hub("newadays/tinyllama_lora_ig_post", token = "") # Online saving
tokenizer.push_to_hub("newadays/tinyllama_lora_ig_post", token = "") # Online saving

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   6%|6         | 6.42MB /  101MB            

Saved model to https://huggingface.co/newadays/tinyllama_lora_ig_post


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...a_ig_post/tokenizer.model: 100%|##########|  500kB /  500kB            

No files have been modified since last commit. Skipping to prevent empty commit.


In [34]:
if True: # Set to False to skip inference testing
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "tinyllama_lora_ig_post", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Generate an creative Instagram caption described in the context provided in triple backticks for a brand called Chernov Team Realtor located in Los Angeles California with emojis.\n\n\n\nPlease follow the steps below to perform the task:\n\n1 - Add important information about the real estate that will captivate readers. Do not add a description about the image.\n\n2 - Itemize the good features of a good property in a List\n\n3 - Include the location of the listing\n\n3 - Remember to add hashtags related to the product at the end of the caption in a separate line.\n\n4 - Not more than 100 words\n\n\n\nPlease remember to follow these important guidelines:\n\n- Remember to use proper grammar throughout the caption\n\n- Keep the caption short and to the point. \n\n- Use an enthusiastic tone throughout the caption.\n\n- Create a Sense of Urgency throughout the caption.\n\n- Remember to itemize the features in a List\n\n\n\n```\n\na white house with the words just listed above it\n\n```", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 64)

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
<s> Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Generate an creative Instagram caption described in the context provided in triple backticks for a brand called Chernov Team Realtor located in Los Angeles California with emojis.



Please follow the steps below to perform the task:

1 - Add important information about the real estate that will captivate readers. Do not add a description about the image.

2 - Itemiz

In [35]:
# Merge to 16bit
if False: model.save_pretrained_merged("tinyllama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/tinyllama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("tinyllama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/tinyllama_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("tinyllama_lora")
    tokenizer.save_pretrained("tinyllama_lora")
if False:
    model.push_to_hub("HF_USERNAME/tinyllama_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/tinyllama_lora", token = "YOUR_HF_TOKEN")

In [36]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("tinyllama_finetune", tokenizer,)
if False: model.push_to_hub_gguf("HF_USERNAME/tinyllama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("tinyllama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/tinyllama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("tinyllama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/tinyllama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")